<div dir="rtl" align="right">

# اختيارُ القنواتِ بِارتباطِ بيرسون

**مجموعةُ البياناتِ**: MOABB BNCI2014-001 (تخيّلٌ حركيّ)  
**القنواتُ**: 22 قناةً  
**معدّلُ أخذِ العيناتِ**: 250 Hz  
**المُشاركُ**: 1

---

## نظرةٌ عامّةٌ

نُزيلُ القنواتِ المُتشابهةَ جداً (|r| > 0.85) لِتقليلِ التكرار.

## المُخرجاتُ المُتوقّعةُ

- مصفوفةُ الارتباطِ بِعلاماتٍ على القنواتِ المَرفوضة
- خريطةُ الرأسِ بِالقنواتِ المُبقاةِ والمَرفوضة

## المُعاملاتُ الأساسيةُ

| المُعاملُ | القيمةُ |
| --- | --- |
| THRESHOLD | 0.85 |

</div>

<div dir="rtl" align="right">

## 1. تثبيتُ المكتباتِ

</div>

In [ ]:
!pip install moabb mne scipy numpy plotly scikit-learn


<div dir="rtl" align="right">

## 2. تحميلُ مجموعةِ بياناتِ MOABB

تُنزّلُ MOABB البياناتِ تلقائيّاً عندَ أوّلِ استدعاءٍ (حوالي 44 ميجابايت).

</div>

In [ ]:
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery
import numpy as np

dataset = BNCI2014_001()
paradigm = MotorImagery(n_classes=2)
X, labels, meta = paradigm.get_data(dataset=dataset, subjects=[1])

mask = (labels == 'left_hand') | (labels == 'right_hand')
X = X[mask]
labels = labels[mask]

print(f'X shape: {X.shape}')
print(f'Labels: {np.unique(labels)}')
print(f'Trials: {len(labels)}')


<div dir="rtl" align="right">

## 3. استكشافُ البياناتِ

</div>

In [ ]:
n_trials, n_channels, n_samples = X.shape
print(f'Trials: {n_trials}')
print(f'Channels: {n_channels}')
print(f'Samples per trial: {n_samples}')
print(f'Trial duration: {n_samples/250:.2f} s')


<div dir="rtl" align="right">

## 4. حسابُ الارتباطِ وإزالةُ التكرارِ

</div>

In [ ]:
THRESHOLD = 0.85

corr_sum = np.zeros((n_channels, n_channels))
for trial in range(n_trials):
    corr = np.corrcoef(X[trial, :, :])
    corr_sum += corr
corr_avg = corr_sum / n_trials

variances = np.var(X.reshape(n_trials, n_channels, n_samples), axis=(0, 2))
to_remove = set()
for i in range(n_channels):
    for j in range(i + 1, n_channels):
        if abs(corr_avg[i, j]) > THRESHOLD:
            if variances[i] < variances[j]:
                to_remove.add(i)
            else:
                to_remove.add(j)

kept = [i for i in range(n_channels) if i not in to_remove]
print(f'Kept {len(kept)} channels, removed {len(to_remove)}')


<div dir="rtl" align="right">

## 5. رسمٌ تفاعليٌّ

**علامَ تُلاحظُ؟**

- القنواتُ المُتجاورةُ تَكونُ عادةً مُرتبطةً ارتباطاً عالياً
- نُبقي القناةَ ذاتَ التباينِ الأعلى (معلوماتٍ أكثر)

</div>

In [ ]:
import plotly.graph_objects as go

fig = go.Figure(data=go.Heatmap(z=corr_avg, colorscale='RdBu_r', zmin=-1, zmax=1))
fig.update_layout(height=600, title='Channel Correlation Matrix', xaxis_title='Channel', yaxis_title='Channel')
fig.show()


<div dir="rtl" align="right">

## خلاصةٌ

- إزالةُ القنواتِ المُكرّرةِ تُقلّلُ الأبعادِ دونَ فقدانِ معلومات
- العتبةُ 0.85 ليستْ مُطلقةً، جَرّبْ قيماً مُختلفة
- طريقةٌ سريعةٌ لا تَتطلّبُ تدريبَ نموذج

</div>